# FG/BG Layer Weights Notebook

**One-time offline computation** on the full ImageNet 1K val set.

For each layer, measures how well the patch map separates foreground (FG) patches
from background (BG) patches in cosine-similarity space:

```
sim_L(patch) = cos( patch_map_L(patch_emb_L), cls_emb_L )

w_L = clip(mean_FG_L - mean_BG_L, 0) / (std_FG_L + std_BG_L + eps)
```

**Inputs:**
- ImageNet val images (any directory layout — flat or synset subdirs)
- Pascal-VOC bbox XML annotations for val set
- Trained patch map checkpoints (`patch_map_full/best_maps`)

**Output:**
- `tmp/layer_avg_weights_patch_map_full.pt` — `{"layers": [...], "weights": [...]}` ready for eval configs

Run from project root:
```bash
source venv/bin/activate
jupyter nbconvert --to notebook --execute notebooks/fg_bg_layer_weights.ipynb
```

In [1]:
%matplotlib inline

import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

sys.path.insert(0, str(Path('..').resolve() / 'src'))

from attribute_lens.scorer import load_patch_map_checkpoint, discover_lens_files
from tuned_lens.bbox_data import parse_bbox_xml, transform_bboxes_224, classify_patches
from tuned_lens.model import VisionModelWrapper
from tuned_lens.config import ModelConfig

print('Imports OK')

Imports OK


## Configuration — edit paths to match your environment

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
IMAGE_DIR     = Path('/opt/models/datasets/imagenet/extracted/val')
BBOX_DIR      = Path('/opt/models/datasets/imagenet/boxes/val')
PATCH_MAP_DIR = Path('/home/dev/attribute_lens/outputs/deit3_large_patch16_224/patch_map_full/best_maps')
OUTPUT_PATH   = Path('/home/dev/attribute_lens/tmp/layer_avg_weights_patch_map_full.pt')

# ── Model selection — uncomment the model you want to use ──────────────────
# MODEL_NAME = 'vit_large_patch14_clip_224.openai_ft_in1k'   # CLIP ViT-L/14
# MODEL_NAME = 'vit_large_patch16_224.augreg_in21k_ft_in1k'  # ViT-L/16 (AugReg)
MODEL_NAME = 'deit3_large_patch16_224.fb_in1k'             # DeiT3-L/16
# MODEL_NAME = 'vit_large_patch14_dinov2.lvd142m'           # DINOv2-L/14    (518×518, 37×37 grid)
# Note: GRID_SIZE and PATCH_SIZE are derived from the model after loading (see Load model cell)

# ── Processing ────────────────────────────────────────────────────────────────
BATCH_SIZE   = 256     # images per forward pass; reduce if OOM
MAX_IMAGES   = None   # None = all val images with bboxes; set e.g. 5000 to limit

# ── FG/BG thresholds (must match training config) ─────────────────────────────
FG_THRESHOLD = 0.80
BG_THRESHOLD = 0.20

EPS          = 1e-6
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device        : {DEVICE}')
print(f'Image dir     : {IMAGE_DIR}')
print(f'Bbox dir      : {BBOX_DIR}')
print(f'Patch map dir : {PATCH_MAP_DIR}')
print(f'Output path   : {OUTPUT_PATH}')

Device        : cuda
Image dir     : /opt/models/datasets/imagenet/extracted/val
Bbox dir      : /opt/models/datasets/imagenet/boxes/val
Patch map dir : /home/dev/attribute_lens/outputs/deit3_large_patch16_224/patch_map_full/best_maps
Output path   : /home/dev/attribute_lens/tmp/layer_avg_weights_patch_map_full.pt


## Discover images with bounding box annotations

Walks `IMAGE_DIR` recursively for JPEG files and matches each image to its bbox XML
by filename stem. Images without a matching XML are skipped.

In [3]:
assert IMAGE_DIR.exists(), f'IMAGE_DIR not found: {IMAGE_DIR}'
assert BBOX_DIR.exists(),  f'BBOX_DIR not found: {BBOX_DIR}'

# Build stem → xml_path index from bbox dir
stem_to_xml = {p.stem: p for p in sorted(BBOX_DIR.glob('*.xml'))}
print(f'Found {len(stem_to_xml)} bbox XMLs in {BBOX_DIR}')

# Walk image dir (handles both flat and synset-subdir layouts)
image_records = []  # list of (img_path, xml_path)
for ext in ('*.JPEG', '*.JPG', '*.jpeg', '*.jpg', '*.png', '*.PNG'):
    for img_path in sorted(IMAGE_DIR.rglob(ext)):
        xml_path = stem_to_xml.get(img_path.stem)
        if xml_path is not None:
            image_records.append((img_path, xml_path))

if MAX_IMAGES is not None:
    image_records = image_records[:MAX_IMAGES]

print(f'Images with bbox annotations: {len(image_records)}')
if image_records:
    print(f'  First: {image_records[0][0].name}')
    print(f'  Last:  {image_records[-1][0].name}')

Found 50000 bbox XMLs in /opt/models/datasets/imagenet/boxes/val
Images with bbox annotations: 50000
  First: ILSVRC2012_val_00000293.JPEG
  Last:  ILSVRC2012_val_00049174.JPEG


## Load model

In [4]:
model_cfg = ModelConfig(
    model_name=MODEL_NAME,
    pretrained=True,
    target_layers=list(range(24)),  # all 24 layers
    freeze_model=True,
    patch_mode=True,
)
wrapper = VisionModelWrapper(model_cfg, device=DEVICE)
wrapper.target_layers = list(range(24))
transform = wrapper.get_transform()
d_model    = wrapper.d_model
GRID_SIZE  = wrapper.patch_grid_size[0]   # 16 for patch14, 14 for patch16
PATCH_SIZE = 224 // GRID_SIZE              # 14 for patch14, 16 for patch16

print(f'Model loaded: {MODEL_NAME}')
print(f'd_model={d_model}  grid={wrapper.patch_grid_size}  patch={PATCH_SIZE}px')

Model loaded: deit3_large_patch16_224.fb_in1k
d_model=1024  grid=(14, 14)  patch=16px


## Load patch maps

In [5]:
assert PATCH_MAP_DIR.exists(), f'Patch map directory not found: {PATCH_MAP_DIR}'

available_maps = discover_lens_files(str(PATCH_MAP_DIR))
print(f'Available patch map layers: {sorted(available_maps.keys())}')

patch_maps = {}
for layer in wrapper.target_layers:
    if layer in available_maps:
        patch_maps[layer] = load_patch_map_checkpoint(available_maps[layer], device=DEVICE)
        patch_maps[layer].eval()
    else:
        print(f'  Layer {layer:2d}: [WARNING] no checkpoint, skipping')

active_layers = sorted(patch_maps.keys())
print(f'Active layers: {active_layers}')

Available patch map layers: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Active layers: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


## Accumulate FG/BG cosine similarities per layer

For each batch of images:
1. Extract patch embeddings and CLS tokens at all layers via `extract_cls_and_patches`
2. Classify patches as FG / BG using bbox annotations
3. Apply the patch map: `mapped = patch_map_L(patch_emb_L)`
4. Compute cosine similarity: `sim = cos(mapped, cls_L)`
5. Accumulate FG and BG similarity values per layer

In [6]:
all_fg_sims = defaultdict(list)  # {layer: [float, ...]}
all_bg_sims = defaultdict(list)
n_skipped = 0

# Process images in batches
for batch_start in tqdm(range(0, len(image_records), BATCH_SIZE), desc='Batches'):
    batch_records = image_records[batch_start: batch_start + BATCH_SIZE]

    # Load images and bboxes
    tensors = []
    batch_bboxes = []
    valid_indices = []

    for idx, (img_path, xml_path) in enumerate(batch_records):
        try:
            img = Image.open(img_path).convert('RGB')
            tensors.append(transform(img))
            bboxes_orig, orig_w, orig_h = parse_bbox_xml(xml_path)
            bboxes_224 = transform_bboxes_224(bboxes_orig, orig_w, orig_h)
            batch_bboxes.append(bboxes_224)
            valid_indices.append(idx)
        except Exception as e:
            print(f'  [WARN] skipping {img_path.name}: {e}')
            continue

    if not tensors:
        continue

    batch_tensor = torch.stack(tensors).to(DEVICE)  # [B, 3, 224, 224]

    # Extract CLS tokens and patch embeddings for all layers in one forward pass
    with torch.no_grad():
        cls_dict, patch_dict, _ = wrapper.extract_cls_and_patches(batch_tensor)
    # cls_dict:   {layer: [B, d_model]}
    # patch_dict: {layer: [B, H, W, d_model]}

    for bi, bboxes_224 in enumerate(batch_bboxes):
        fg_mask, bg_mask = classify_patches(
            bboxes_224,
            grid_size=GRID_SIZE,
            patch_size=PATCH_SIZE,
            fg_threshold=FG_THRESHOLD,
            bg_threshold=BG_THRESHOLD,
        )
        fg_flat = torch.from_numpy(fg_mask.reshape(-1))  # [H*W] bool
        bg_flat = torch.from_numpy(bg_mask.reshape(-1))

        if not fg_flat.any() and not bg_flat.any():
            n_skipped += 1
            continue

        for layer in active_layers:
            # patches: [H, W, d] → [H*W, d]
            patches_flat = patch_dict[layer][bi].reshape(-1, d_model).float()
            cls_emb      = cls_dict[layer][bi].float()  # [d_model]

            with torch.no_grad():
                mapped = patch_maps[layer](patches_flat)  # [H*W, d]

            mapped_n = F.normalize(mapped, p=2, dim=-1)                    # [H*W, d]
            cls_n    = F.normalize(cls_emb.unsqueeze(0), p=2, dim=-1)     # [1, d]
            sims     = (mapped_n * cls_n).sum(dim=-1).cpu()               # [H*W]

            if fg_flat.any():
                all_fg_sims[layer].extend(sims[fg_flat].tolist())
            if bg_flat.any():
                all_bg_sims[layer].extend(sims[bg_flat].tolist())

print(f'Done. Skipped {n_skipped} images with no FG or BG patches.')
print('\nPatch counts per layer:')
for layer in active_layers:
    print(f'  Layer {layer:2d}: {len(all_fg_sims[layer]):6d} FG, {len(all_bg_sims[layer]):6d} BG')

Batches: 100%|██████████| 196/196 [1:40:54<00:00, 30.89s/it]  

Done. Skipped 0 images with no FG or BG patches.

Patch counts per layer:
  Layer  0: 4869137 FG, 4141123 BG
  Layer  1: 4869137 FG, 4141123 BG
  Layer  2: 4869137 FG, 4141123 BG
  Layer  3: 4869137 FG, 4141123 BG
  Layer  4: 4869137 FG, 4141123 BG
  Layer  5: 4869137 FG, 4141123 BG
  Layer  6: 4869137 FG, 4141123 BG
  Layer  7: 4869137 FG, 4141123 BG
  Layer  8: 4869137 FG, 4141123 BG
  Layer  9: 4869137 FG, 4141123 BG
  Layer 10: 4869137 FG, 4141123 BG
  Layer 11: 4869137 FG, 4141123 BG
  Layer 12: 4869137 FG, 4141123 BG
  Layer 13: 4869137 FG, 4141123 BG
  Layer 14: 4869137 FG, 4141123 BG
  Layer 15: 4869137 FG, 4141123 BG
  Layer 16: 4869137 FG, 4141123 BG
  Layer 17: 4869137 FG, 4141123 BG
  Layer 18: 4869137 FG, 4141123 BG
  Layer 19: 4869137 FG, 4141123 BG
  Layer 20: 4869137 FG, 4141123 BG
  Layer 21: 4869137 FG, 4141123 BG
  Layer 22: 4869137 FG, 4141123 BG
  Layer 23: 4869137 FG, 4141123 BG


## Compute w_L per layer

$$w_L = \frac{\max(\overline{\text{sim}}_{\text{FG},L} - \overline{\text{sim}}_{\text{BG},L},\, 0)}{\sigma_{\text{FG},L} + \sigma_{\text{BG},L} + \varepsilon}$$

In [7]:
weights = {}
rows = []

print(f'{"Layer":>6} {"mean_FG":>10} {"mean_BG":>10} {"std_FG":>10} {"std_BG":>10} {"weight":>10}')
print('-' * 62)

for layer in active_layers:
    fg_vals = all_fg_sims[layer]
    bg_vals = all_bg_sims[layer]

    if len(fg_vals) == 0 or len(bg_vals) == 0:
        print(f'  Layer {layer:2d}: insufficient data, setting w=0')
        weights[layer] = 0.0
        rows.append((layer, float('nan'), float('nan'), float('nan'), float('nan'), 0.0))
        continue

    fg_t = torch.tensor(fg_vals)
    bg_t = torch.tensor(bg_vals)

    mean_fg = float(fg_t.mean())
    mean_bg = float(bg_t.mean())
    std_fg  = float(fg_t.std()) if len(fg_vals) > 1 else 0.0
    std_bg  = float(bg_t.std()) if len(bg_vals) > 1 else 0.0

    numerator   = max(mean_fg - mean_bg, 0.0)
    denominator = std_fg + std_bg + EPS
    w = numerator / denominator

    weights[layer] = w
    rows.append((layer, mean_fg, mean_bg, std_fg, std_bg, w))
    print(f'{layer:>6} {mean_fg:>10.4f} {mean_bg:>10.4f} {std_fg:>10.4f} {std_bg:>10.4f} {w:>10.4f}')

print('-' * 62)
print(f'\nWeights: { {l: round(w, 4) for l, w in weights.items()} }')

 Layer    mean_FG    mean_BG     std_FG     std_BG     weight
--------------------------------------------------------------
     0     0.9986     0.9913     0.0214     0.0569     0.0933
     1     0.9987     0.9921     0.0181     0.0482     0.0990
     2     0.9989     0.9937     0.0135     0.0359     0.1055
     3     0.9991     0.9949     0.0101     0.0270     0.1119
     4     0.9992     0.9957     0.0079     0.0210     0.1214
     5     0.9992     0.9954     0.0082     0.0223     0.1238
     6     0.9992     0.9960     0.0066     0.0182     0.1322
     7     0.9992     0.9960     0.0065     0.0184     0.1322
     8     0.9993     0.9964     0.0055     0.0161     0.1337
     9     0.9994     0.9973     0.0037     0.0115     0.1378
    10     0.9996     0.9989     0.0011     0.0037     0.1566
    11     0.9995     0.9985     0.0011     0.0038     0.1929
    12     0.9993     0.9979     0.0012     0.0044     0.2512
    13     0.9989     0.9971     0.0012     0.0044     0.3227
    14 

## Visualize

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

layer_labels = [str(l) for l in active_layers]
w_vals    = [weights[l] for l in active_layers]
mean_fg_v = [r[1] for r in rows]
mean_bg_v = [r[2] for r in rows]

axes[0].bar(layer_labels, w_vals, color='steelblue')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('w_L')
axes[0].set_title('Layer weights (FG–BG discriminability)')
axes[0].tick_params(axis='x', rotation=45)

x = np.arange(len(active_layers))
axes[1].plot(x, mean_fg_v, 'o-', color='#E45756', label='mean_FG')
axes[1].plot(x, mean_bg_v, 's-', color='#4C78A8', label='mean_BG')
axes[1].set_xticks(x)
axes[1].set_xticklabels(layer_labels, rotation=45)
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Mean cosine similarity')
axes[1].set_title('Mean FG / BG cosine similarity per layer')
axes[1].legend()

gap = [mf - mb for mf, mb in zip(mean_fg_v, mean_bg_v)]
colors = ['#E45756' if g > 0 else '#4C78A8' for g in gap]
axes[2].bar(layer_labels, gap, color=colors)
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].set_xlabel('Layer')
axes[2].set_ylabel('mean_FG - mean_BG')
axes[2].set_title('FG−BG gap per layer')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [9]:
n_show = min(6, len(active_layers))
step   = max(1, len(active_layers) // n_show)
show_layers = active_layers[::step][:n_show]

fig, axes = plt.subplots(1, len(show_layers), figsize=(4 * len(show_layers), 4))
if len(show_layers) == 1:
    axes = [axes]

for ax, layer in zip(axes, show_layers):
    bins = np.linspace(-1, 1, 60)
    ax.hist(all_bg_sims[layer][:5000], bins=bins, alpha=0.6, color='#4C78A8', label='BG', density=True)
    ax.hist(all_fg_sims[layer][:5000], bins=bins, alpha=0.6, color='#E45756', label='FG', density=True)
    ax.set_title(f'Layer {layer}\nw={weights[layer]:.3f}', fontsize=9)
    ax.set_xlabel('cos(mapped_patch, cls)')
    if ax is axes[0]:
        ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('FG / BG cosine similarity distributions (sample of layers)', fontsize=11)
plt.tight_layout()
plt.show()

## Save weights

In [10]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

output = {
    'layers':  active_layers,
    'weights': [weights[l] for l in active_layers],
    'metadata': {
        'image_dir':    str(IMAGE_DIR),
        'bbox_dir':     str(BBOX_DIR),
        'patch_map_dir': str(PATCH_MAP_DIR),
        'n_images':     len(image_records),
        'fg_threshold': FG_THRESHOLD,
        'bg_threshold': BG_THRESHOLD,
        'eps':          EPS,
        'formula':      'clip(mean_FG - mean_BG, 0) / (std_FG + std_BG + eps)',
    }
}

torch.save(output, OUTPUT_PATH)
print(f'Saved to: {OUTPUT_PATH}')
print(f'  layers : {output["layers"]}')
print(f'  weights: {[round(w, 4) for w in output["weights"]]}')
print()
print('Top-3 layers by weight:')
for layer, w in sorted(weights.items(), key=lambda x: x[1], reverse=True)[:3]:
    print(f'  Layer {layer}: w={w:.4f}')

Saved to: /home/dev/attribute_lens/tmp/layer_avg_weights_patch_map_full.pt
  layers : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
  weights: [0.0933, 0.099, 0.1055, 0.1119, 0.1214, 0.1238, 0.1322, 0.1322, 0.1337, 0.1378, 0.1566, 0.1929, 0.2512, 0.3227, 0.4408, 0.2901, 0.3263, 0.3635, 0.3623, 0.3195, 0.2658, 0.2195, 0.1858, 0.1457]

Top-3 layers by weight:
  Layer 14: w=0.4408
  Layer 17: w=0.3635
  Layer 18: w=0.3623


## How to use in experiment configs

The saved file is already referenced in the weighted layer-avg experiment configs:
```
configs/experiments/exp_no_nb_la_weighted.yaml
configs/experiments/exp_nb_emb_la_weighted.yaml
configs/experiments/exp_nb_score_la_weighted.yaml
```

Each of those has:
```yaml
eval:
  layer_avg:
    enabled: true
    weights_path: "tmp/layer_avg_weights_patch_map_full.pt"
```